# EDA & Feature Engineering Notebook

**Two modes:**
- **Interactive** — run each cell individually, inspect results, decide which fixes to apply
- **Automated** — run Section 10 only to produce a clean CSV without intervention

Every EDA function returns an `EDAResult` that tells you:
1. What the issue is
2. How severe it is
3. Which `feat_engineering.py` function fixes it

---

## 0. Setup

In [23]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Add project root to path
sys.path.append(str(Path.cwd().parent.parent))

import eda_functions as eda
import feat_engineering as fe

# ── CONFIG — edit these for your project ────────────────────────────────────
DATA_PATH    = 'data/raw/synthetic_car_prices.csv'   # path to raw data
TARGET       = 'actual_euro'                        # target column name
TASK         = 'regression'                   # 'regression' | 'classification' | 'deep_learning'
DATE_COLS    = ['registration_date', 'sold_at']  # date columns (or [])
LEAKY_COLS   = []                             # known leaky columns to drop immediately
OUTPUT_DIR   = 'data/processed'
OUTPUT_FILE  = 'clean_dataset.csv'
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

Loaded: 500 rows x 10 columns


In [24]:
df.head()

,actual_euro,predicted_euro,fuel,car_type,paint_color,mileage,car_age_years,engine_power,annual_mileage,date
0,17297,15633,Diesel,Hatchback,Blue,17804,4.923255,144,17007,2021-04-01 19:36:26.917668691
1,22950,24670,Electric,SUV,Grey,75193,5.772790,149,17341,2022-09-07 04:09:24.435668302
2,16628,16842,Petrol,SUV,Silver,105277,8.655771,218,17599,2021-12-01 22:12:59.637897584
3,14406,13540,Petrol,Coupe,Red,45832,9.807393,240,10110,2023-08-16 16:21:39.245655623
4,15715,14809,Diesel,Hatchback,Black,6975,4.075842,167,11588,2023-05-19 13:27:22.217698592


## 1. Dataset Overview

`eda.overview()` — detects MISSING_VALUES → fix with `fe.fix_missing_values()`

In [25]:
result = eda.overview(df, target=TARGET)
result.print_report()


DATASET OVERVIEW
  Rows              : 500
  Columns           : 10
  Memory usage      : 0.15 MB

  Dtypes breakdown:
    int64                5 columns
    object               4 columns
    float64              1 columns

  Missing values    : none

  Target 'actual_euro':
    mean=19469.64  std=5020.93  min=5747.00  max=36858.00

✅ [MISSING_VALUES] — severity: OK
   Affected : none
   Summary  : No missing values detected.
   Fix with : feat_engineering.fix_missing_values()


In [26]:
# Apply fix if needed
# Strategies: 'auto' | 'median' | 'mode' | 'drop_rows' | 'constant'
if result.severity != 'ok':
    df = fe.fix_missing_values(df, strategy='auto', target=TARGET)
    print(f'Shape after fix: {df.shape}')

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   actual_euro     500 non-null    int64  
 1   predicted_euro  500 non-null    int64  
 2   fuel            500 non-null    object 
 3   car_type        500 non-null    object 
 4   paint_color     500 non-null    object 
 5   mileage         500 non-null    int64  
 6   car_age_years   500 non-null    float64
 7   engine_power    500 non-null    int64  
 8   annual_mileage  500 non-null    int64  
 9   date            500 non-null    object 
dtypes: float64(1), int64(5), object(4)
memory usage: 39.2+ KB


## 2. Schema Validation

`eda.validate_schema()` — detects INCONSISTENT_TYPES, HIGH_CARDINALITY

In [27]:
# Optional: define expected schema
# expected_schema = {'price': 'float', 'mileage': 'int', 'fuel': 'object'}
expected_schema = {'actual_euro': 'float','predicted_euro':'float','fuel':'str','car_type':'str','oops':'int'}

results = eda.validate_schema(df, expected_schema=expected_schema,
                               cardinality_threshold=10)
for r in results:
    r.print_report()


SCHEMA VALIDATION
  ⚠️  actual_euro                         expected=float        actual=int64
  ⚠️  predicted_euro                      expected=float        actual=int64
  ⚠️  fuel                                expected=str          actual=object
  ⚠️  car_type                            expected=str          actual=object
  ❌ Column 'oops' expected but not found.

  Cardinality check (threshold=10):
  ✅ fuel                                    4 unique values
  ✅ car_type                                5 unique values
  ✅ paint_color                             6 unique values
  ⚠️  date                                  500 unique values

❌ [INCONSISTENT_TYPES] — severity: CRITICAL
   Affected : ['actual_euro', 'predicted_euro', 'fuel', 'car_type', 'oops']
   Summary  : 5 columns have unexpected types.
   Fix with : feat_engineering.fix_dtypes()

⚠️  [HIGH_CARDINALITY] — severity: WARNING
   Affected : ['date']
   Summary  : 1 categorical columns exceed 10 unique values.
   Fix wit

In [ ]:
# Fix type issues if any
# df = fe.fix_dtypes(df, type_map={'price': 'float', 'sold_at': 'datetime'})

# Fix high cardinality
high_card_result = [r for r in results if r.issue_code == 'HIGH_CARDINALITY' and r.severity != 'ok']
if high_card_result:
    cols = high_card_result[0].affected_columns
    method = 'target' if TASK == 'regression' else 'frequency'
    df = fe.fix_high_cardinality(df, columns=cols, method=method, target=TARGET)
    print(f'Shape after encoding: {df.shape}')

## 3. Data Quality

`eda.check_data_quality()` — detects DUPLICATE_ROWS, CONSTANT_FEATURE, NEAR_DUPLICATE_COLS

In [29]:
results = eda.check_data_quality(df)
for r in results:
    r.print_report()


DATA QUALITY CHECK
  ✅ Duplicate rows    : 0 (0.00%)

  Variance check (constant features):

  Near-duplicate columns (r > 0.99): 0 pairs

✅ [DUPLICATE_ROWS] — severity: OK
   Affected : none
   Summary  : 0 duplicate rows detected (0.00%).
   Fix with : feat_engineering.fix_duplicates()
   Details  :
     n_duplicates: 0
     pct: 0.0000


In [ ]:
# Fix duplicates
df = fe.fix_duplicates(df)

# Fix constant features
df = fe.fix_low_variance(df, target=TARGET)

# Fix near-duplicate columns — specify which pairs to keep
# near_dup = [r for r in results if r.issue_code == 'NEAR_DUPLICATE_COLS']
# if near_dup:
#     df = fe.fix_near_duplicate_columns(df, pairs=near_dup[0].details['pairs'])

print(f'Shape after quality fixes: {df.shape}')

## 4. Distribution Analysis

`eda.check_distributions()` — detects HIGH_SKEWNESS, RARE_CATEGORIES, CLASS_IMBALANCE

In [30]:
results = eda.check_distributions(df)
for r in results:
    r.print_report()


DISTRIBUTION ANALYSIS

  Skewness (numeric columns):
  ✅ actual_euro                         skew=+0.103
  ✅ predicted_euro                      skew=+0.057
  ⚠️  mileage                             skew=+1.214
  ✅ car_age_years                       skew=-0.054
  ✅ engine_power                        skew=-0.009
  ✅ annual_mileage                      skew=+0.051

  Rare categories (categorical columns):
  ✅ fuel                                no rare categories
  ✅ car_type                            no rare categories
  ✅ paint_color                         no rare categories
  ⚠️  date                                500 rare categories: ['2021-04-01 19:36:26.917668691', '2020-12-15 12:47:18.489081567', '2022-06-30 21:13:49.010979675', '2023-01-04 06:43:36.729981853', '2020-02-09 09:24:01.727567957']

⚠️  [HIGH_SKEWNESS] — severity: WARNING
   Affected : ['mileage']
   Summary  : 1 numeric columns are skewed (|skew| > 1.0).
   Fix with : feat_engineering.fix_skewness()
   Details  

In [ ]:
# Fix rare categories
df = fe.fix_rare_categories(df, threshold=0.02)

# Fix skewness in features (NOT target — handled separately in Section 8)
df, transform_map = fe.fix_skewness(df, skew_threshold=1.0, target=TARGET)
print(f'Transforms applied: {transform_map}')

# Fix class imbalance (classification only)
if TASK == 'classification':
    df = fe.fix_class_imbalance(df, target=TARGET, method='smote')

print(f'Shape after distribution fixes: {df.shape}')

## 5. Diversity Check

`eda.check_diversity()` — detects LOW_DIVERSITY, DATE_RANGE_TOO_NARROW

In [32]:
df['date'] = pd.to_datetime(df['date'])
results = eda.check_diversity(df, date_cols=['date'], min_date_range_years=2.0)
for r in results:
    r.print_report()


DIVERSITY CHECK

  Date range check:
  ✅ date                                range=5.0 years (2020-01-08 → 2024-12-29)

  Numeric diversity (unique value ratio):
  ✅ actual_euro                         unique_ratio=0.9780
  ✅ predicted_euro                      unique_ratio=0.9840
  ✅ mileage                             unique_ratio=0.9920
  ✅ car_age_years                       unique_ratio=1.0000
  ✅ engine_power                        unique_ratio=0.3580
  ✅ annual_mileage                      unique_ratio=0.9840

  Categorical diversity (Shannon entropy):
  ✅ fuel                                normalised_entropy=0.857
  ✅ car_type                            normalised_entropy=0.916
  ✅ paint_color                         normalised_entropy=1.000

✅ [LOW_DIVERSITY] — severity: OK
   Affected : none
   Summary  : All features show acceptable diversity.
   Fix with : feat_engineering.fix_low_variance()


In [ ]:
# Extract temporal features from date columns
if DATE_COLS:
    df = fe.fix_date_diversity(df, date_cols=DATE_COLS)
    print(f'Shape after date extraction: {df.shape}')

## 6. Outlier Detection

`eda.detect_outliers()` — detects OUTLIERS → fix with `fe.fix_outliers()`

In [ ]:
# method: 'iqr' (robust) or 'zscore' (assumes normality)
result = eda.detect_outliers(df, method='iqr', iqr_factor=1.5)
result.print_report()

In [ ]:
if result.severity != 'ok':
    # method: 'cap' (Winsorise) | 'remove' | 'log'
    df = fe.fix_outliers(df, method='cap', iqr_factor=1.5, target=TARGET)
    print(f'Shape after outlier treatment: {df.shape}')

## 7. Correlation & Interaction Analysis

`eda.analyse_correlations()` — detects DATA_LEAKAGE_RISK, INTERACTION_SIGNAL

In [28]:
results = eda.analyse_correlations(df, target=TARGET)
for r in results:
    r.print_report()


CORRELATION ANALYSIS

  Feature correlations with target 'actual_euro':
  ❌ predicted_euro                      |r|=0.9593
  ✅ car_age_years                       |r|=0.4625
  ✅ engine_power                        |r|=0.4303
  ✅ mileage                             |r|=0.4146
  ✅ annual_mileage                      |r|=0.3212

  Feature-feature correlation matrix (top pairs):

❌ [DATA_LEAKAGE_RISK] — severity: CRITICAL
   Affected : ['predicted_euro']
   Summary  : 1 features have |r| > 0.95 with target — likely data leakage.
   Fix with : feat_engineering.fix_data_leakage()
   Details  :
     predicted_euro: 0.9593


In [21]:
# Drop leaky features (if any detected or known in advance)
leaky = [r for r in results if r.issue_code == 'predicted_euro']
all_leaky = LEAKY_COLS + (leaky[0].affected_columns if leaky else [])
if all_leaky:
    df = fe.fix_data_leakage(df, columns=all_leaky)

# Create interaction features
# Option A: auto-select pairs based on target correlation
df = fe.create_interaction_features(df, target=TARGET, auto_select=True, top_n=8)

# Option B: specify pairs manually
# df = fe.create_interaction_features(df,
#     pairs=[('car_age_years', 'mileage'), ('engine_power', 'car_age_years')],
#     operations=['multiply', 'divide'])

print(f'Shape after correlation fixes: {df.shape}')

KeyError: 'predicted_price'

## 8. Multicollinearity

`eda.detect_multicollinearity()` — detects MULTICOLLINEARITY → fix with `fe.fix_multicollinearity()`

In [ ]:
result = eda.detect_multicollinearity(df, target=TARGET, vif_threshold=10.0)
result.print_report()

In [ ]:
if result.severity != 'ok':
    # method: 'vif' (iterative VIF drop) | 'correlation' (pairwise r > 0.9)
    df = fe.fix_multicollinearity(df, target=TARGET, vif_threshold=10.0)
    print(f'Shape after multicollinearity fix: {df.shape}')

## 9. Target Variable Transformation

Apply AFTER all feature engineering. Required for regression with skewed targets.
Keep the `inverse_fn` — you MUST apply it to predictions before computing business metrics.

In [ ]:
if TASK in ('regression', 'deep_learning'):
    print(f'Target skewness before transform: {df[TARGET].skew():.4f}')
    
    # method: 'log1p' | 'sqrt' | 'box-cox' | 'yeo-johnson'
    df, inverse_fn = fe.fix_target_transform(df, target=TARGET, method='log1p')
    
    print(f'Target skewness after transform : {df[TARGET].skew():.4f}')
    print('\n⚠️  Remember: predictions = inverse_fn(model.predict(X))')
    print('   For log1p: predictions = np.expm1(model.predict(X))')

## 10. Residual Analysis (post-training)

Run this AFTER training your model to check residual quality.

In [ ]:
# Uncomment and fill in after training:
# from src.models.train import load_model
# model = load_model('data/models/checkpoints/your_model.pkl')
# y_pred = model.predict(X_val)
# results = eda.analyse_residuals(y_val, y_pred)
# for r in results:
#     r.print_report()

## 11. Save Clean Dataset

In [ ]:
path = fe.save_clean_dataset(
    df,
    filename=OUTPUT_FILE,
    output_dir=OUTPUT_DIR,
    also_save_metadata=True   # saves a JSON pipeline log alongside the CSV
)
print(f'\n✅ Clean dataset saved to: {path}')
print(f'   Ready to use in train.py')

---
## AUTOMATED MODE

Run the cell below to skip all interactive steps and produce the clean dataset in one shot.
Configure the parameters at the top of Section 0 before running.

In [ ]:
# ── AUTOMATED PIPELINE — runs everything end-to-end ──────────────────────────
# Reload raw data to start from scratch
df_raw = pd.read_csv(DATA_PATH)

df_clean, output_path = fe.run_full_pipeline(
    df=df_raw,
    target=TARGET,
    task=TASK,
    date_cols=DATE_COLS,
    leaky_cols=LEAKY_COLS,
    interaction_pairs=None,        # None = auto-select
    output_dir=OUTPUT_DIR,
    output_filename=OUTPUT_FILE,
    skew_threshold=1.0,
    outlier_method='cap',
    cardinality_threshold=50,
    missing_strategy='auto',
    target_transform='log1p',      # None to skip
)

print(f'\n✅ Pipeline complete. Clean dataset: {output_path}')
print(f'   Shape: {df_clean.shape}')